# SFT (Colab Pro) — 1 epoch, eval every 1k steps, test at end

Thin launcher: all logic lives in `ml/train.py`. This notebook only fetches data and runs it.

**GPU:** pick **A100** (Runtime > Change runtime type > A100). SFT is GPU-bound (no reward VM),
so the fast GPU pays off here — unlike the RL notebook. A100/L4 also have native bf16; **avoid T4**
(no native bf16, the whole stack is bf16).

**Secrets** (Colab key icon, toggle Notebook access on): `GITHUB_TOKEN`, `WANDB_API_KEY`.
Add `HF_TOKEN` only if the dataset repo is private. No reward server / ngrok / VM needed for SFT.

**Data:** input dataset is pulled from HuggingFace (`Strhata/ebpf-corpus` →
`dataset_final_qwen_enriched.jsonl`) to local SSD. Output checkpoints go to Drive so they
survive the 24h session kill.

In [ ]:
# Cell 1 — Mount Google Drive (input dataset + output checkpoints both live here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Clone repo on first run, git pull on restarts (idempotent)
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
repo_dir = '/content/ebpf-fuzzing-thesis'

if not os.path.exists(repo_dir):
    ret = os.system(f'git clone https://{token}@github.com/Strhata/ebpf-fuzzing-thesis.git {repo_dir}')
    assert ret == 0, 'git clone failed'
else:
    ret = os.system(f'git -C {repo_dir} pull')
    assert ret == 0, 'git pull failed'

os.chdir(repo_dir)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3 — Install training-side dependencies (~10 min on a fresh runtime)
import subprocess
subprocess.run(
    ['pip', 'install', '-q', '-r', 'ml/requirements_colab.txt'],
    check=True,
)

In [ ]:
# Cell 4 — Config + pull dataset from HuggingFace (the only cell you may need to edit)
# RE-RUN THIS CELL after any edit, then run Cell 5.
import os
from huggingface_hub import hf_hub_download

# --- where things live ---
HF_REPO     = 'Strhata/ebpf-corpus'                      # dataset lives here (HF CDN, fast)
HF_FILE     = 'dataset_final_qwen_enriched.jsonl'        # the enriched file train.py needs
RUN_NAME    = 'sft-1epoch-v2'
OUTPUT_DIR  = f'/content/drive/MyDrive/{RUN_NAME}'       # checkpoints survive the 24h session death

# --- training knobs ---
EPOCHS      = 1
EVAL_STEPS  = 1000   # eval + checkpoint + encoder pass-rate every N optimizer steps
MAX_STEPS   = 0      # 0 = full run. Set e.g. 30 for a quick smoke to measure it/s first.

# --- download dataset HF -> local SSD (cached; re-runs instant). Add HF_TOKEN secret only if repo is private. ---
from google.colab import userdata
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass  # public repo needs no token
print('[*] Fetching dataset from HuggingFace ...')
DATA_LOCAL = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, repo_type='dataset')
print(f'DATA={DATA_LOCAL}  ({os.path.getsize(DATA_LOCAL)/1e9:.2f} GB)')
print(f'OUTPUT_DIR={OUTPUT_DIR}')
print(f'EPOCHS={EPOCHS}  EVAL_STEPS={EVAL_STEPS}  MAX_STEPS={MAX_STEPS or "full"}  RUN_NAME={RUN_NAME}')

In [ ]:
# Cell 5 — Launch SFT (safe to re-run: --resume continues from the last Drive checkpoint)
import os
from google.colab import userdata

# --- WandB: train.py does wandb.init(project="ebpf-thesis", name=RUN_NAME) + report_to="wandb" ---
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')   # 401s here if the secret is missing
os.environ['WANDB_PROJECT'] = 'ebpf-thesis'
import wandb; wandb.login()   # fail fast + print the logged-in entity before the long run
print(f'[*] WandB ready -> project=ebpf-thesis  run={RUN_NAME}')

max_steps_flag = f' --max-steps {MAX_STEPS}' if MAX_STEPS and MAX_STEPS > 0 else ''

cmd = (
    f'python ml/train.py'
    f' --data {DATA_LOCAL}'
    f' --output-dir {OUTPUT_DIR}'
    f' --epochs {EPOCHS}'
    f' --eval-steps {EVAL_STEPS}'
    f' --run-name {RUN_NAME}'
    f' --resume'
    f'{max_steps_flag}'
)
print('LAUNCH:', cmd)
os.system(cmd)